Turn scored study and recall records into a dataset and aligned semantic inputs. This small, invented example uses four stimuli and three lists so that every conversion is visible; its feature values are for teaching, not empirical semantic estimates. Run the notebook from top to bottom after [Developer Setup](../developer-setup.ipynb), or use Colab; no input files are needed.

In [1]:
#| code-summary: "1 — Set the output location"

project_root = None  # Git root locally; the working directory otherwise.
output_dir = "results/tutorial/data-preparation"


In [2]:
#| code-summary: "2 — Install JAXCMR when running in Colab"

import sys

if "google.colab" in sys.modules:
    import subprocess

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "jaxcmr @ git+https://github.com/githubpsyche/jaxcmr.git@main",
    ])


In [3]:
#| code-summary: "3 — Import packages"

import json
from pathlib import Path

import jax.numpy as jnp
import numpy as np

from jaxcmr.components.factory import build_trial_connections_from_similarity
from jaxcmr.helpers import (
    find_project_root,
    load_data,
    resolve_project_path,
    save_dict_to_hdf5,
)
from jaxcmr.math import cosine_similarity_matrix


In [4]:
#| code-summary: "4 — Resolve the working folder"

if project_root is not None:
    root_path = resolve_project_path(project_root, Path.cwd())
else:
    try:
        root_path = Path(find_project_root())
    except FileNotFoundError:
        root_path = Path.cwd()
output_path = resolve_project_path(output_dir, root_path)
output_path.mkdir(parents=True, exist_ok=True)


## Start with scored records

Each record is one study–recall trial. Responses have already been matched to stimulus identities, in recall order. These lists contain no repeated presentations or intrusions; deciding how to score those is part of preparing a real experiment, before this conversion.

In [5]:
#| code-summary: "5 — Describe the study lists and scored recall order"

records = [
    {"subject": 1, "condition": 1,
     "studied": ["apple", "pear", "hammer"], "recalled": ["hammer", "apple"]},
    {"subject": 1, "condition": 2,
     "studied": ["cup", "hammer", "apple", "pear"], "recalled": ["pear", "cup"]},
    {"subject": 2, "condition": 1,
     "studied": ["pear", "apple", "cup"], "recalled": []},
]

stimuli = ["apple", "pear", "hammer", "cup"]
item_ids = {name: index + 1 for index, name in enumerate(stimuli)}
print("Shared stimulus IDs:", item_ids)


Shared stimulus IDs: {'apple': 1, 'pear': 2, 'hammer': 3, 'cup': 4}


## Align trials, study positions and item identities

Positive `recalls` codes identify **one-based study positions**; zero is padding. `pres_itemids` holds identities shared across trials, while `pres_itemnos` holds within-list item codes. With distinct items in each list, the latter equal study positions. An empty recall sequence stays an all-zero row, retaining that trial and participant.

In [6]:
#| code-summary: "6 — Convert stimulus identities into aligned arrays"

trial_count = len(records)
study_width = max(len(record["studied"]) for record in records)
response_width = study_width
pres_itemnos = np.zeros((trial_count, study_width), dtype=np.int32)
pres_itemids = np.zeros_like(pres_itemnos)
recalls = np.zeros((trial_count, response_width), dtype=np.int32)
rec_itemids = np.zeros_like(recalls)

for trial_index, record in enumerate(records):
    study_positions = {item: position for position, item in enumerate(record["studied"], 1)}
    for column, item in enumerate(record["studied"]):
        pres_itemnos[trial_index, column] = column + 1
        pres_itemids[trial_index, column] = item_ids[item]
    for output_position, item in enumerate(record["recalled"]):
        recalls[trial_index, output_position] = study_positions[item]
        rec_itemids[trial_index, output_position] = item_ids[item]

data = {
    "subject": np.array([[record["subject"]] for record in records], dtype=np.int32),
    "listLength": (pres_itemnos > 0).sum(axis=1, keepdims=True).astype(np.int32),
    "pres_itemnos": pres_itemnos,
    "recalls": recalls,
    "pres_itemids": pres_itemids,
    "rec_itemids": rec_itemids,
    "condition": np.array([[record["condition"]] for record in records], dtype=np.int32),
}
for name in ["listLength", "pres_itemids", "recalls"]:
    print(name, "=\n", data[name])


listLength =
 [[3]
 [4]
 [3]]
pres_itemids =
 [[1 2 3 0]
 [4 3 1 2]
 [2 1 4 0]]
recalls =
 [[3 1 0 0]
 [4 1 0 0]
 [0 0 0 0]]


In the first trial, `hammer` is shared item 3 at study position 3; in the second, it is still shared item 3 but at study position 2. The second trial's first recall, `pear`, is therefore stored as position **4**, not its shared item ID **2**. See [Data](../overview/data.ipynb) for the field reference and repeated-item conventions.

In [7]:
#| code-summary: "7 — Check that recall positions recover the scored identities"

for trial_index, record in enumerate(records):
    positions = data["recalls"][trial_index]
    recalled_ids = data["pres_itemids"][trial_index, positions[positions > 0] - 1]
    expected_ids = [item_ids[item] for item in record["recalled"]]
    assert recalled_ids.tolist() == expected_ids
print("Recall order and stimulus identities preserved in all three trials.")


Recall order and stimulus identities preserved in all three trials.


## Prepare semantic inputs

Rows of an item-feature matrix must follow the shared ID order: item 1 is row 0. Here the features are supplied explicitly; real features could come from a separately chosen embedding model or stimulus descriptors. Human pairwise similarity ratings instead begin with an item-by-item matrix and its row/column labels.

In [8]:
#| code-summary: "8 — Align item features and calculate cosine similarities"

features_by_item = {
    "apple": [1.0, 0.0, 0.0],
    "pear": [1.0, 1.0, 0.0],
    "hammer": [0.0, 0.0, 1.0],
    "cup": [0.0, 1.0, 1.0],
}
features = jnp.asarray([features_by_item[item] for item in stimuli])
similarities = cosine_similarity_matrix(features)
# For cosine similarity specifically, cosine distance is 1 - similarity.
distances = 1.0 - similarities
print("Row and column order:", stimuli)
print("Cosine similarities:\n", np.round(similarities, 3))
print("Cosine distances:\n", np.round(distances, 3))


Row and column order: ['apple', 'pear', 'hammer', 'cup']
Cosine similarities:
 [[1.    0.707 0.    0.   ]
 [0.707 1.    0.    0.5  ]
 [0.    0.    1.    0.707]
 [0.    0.5   0.707 1.   ]]
Cosine distances:
 [[0.    0.293 1.    1.   ]
 [0.293 0.    1.    0.5  ]
 [1.    1.    0.    0.293]
 [1.    0.5   0.293 0.   ]]


A larger similarity means a closer pair; a smaller distance means a closer pair. The conversion `1 - similarity` here is specific to cosine distance, not a general rule for arbitrary rating scales. Features, similarities and distances are separate files, and none belongs in the trial-aligned dataset dictionary.

In [9]:
#| code-summary: "9 — Extract model associations in each trial’s study order"

trial_connections = build_trial_connections_from_similarity(
    data["pres_itemids"], similarities
)
print("First trial:", records[0]["studied"], "+ one padded slot")
print(np.round(trial_connections[0], 3))
assert np.all(np.asarray(trial_connections[0, -1]) == 0)
assert np.all(np.asarray(trial_connections[0, :, -1]) == 0)


First trial: ['apple', 'pear', 'hammer'] + one padded slot
[[0.    0.707 0.    0.   ]
 [0.707 0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.    0.    0.    0.   ]]


This model helper removes self-connections, clips negative similarities, and excludes padding. Pass the original feature matrix to a feature-based semantic model such as `additive_semantic_cmr`, or the similarity matrix to `additive_semantic_matrix_cmr`; do not give a similarity matrix to a factory expecting feature vectors. The [distance-CRP](../templates/distcrp.ipynb) and [distance-rank](../templates/distrank.ipynb) templates accept distances for describing recall organization. The three supplied workshop fits do not include a semantic input.

## Save and reuse the inputs

In [10]:
#| code-summary: "10 — Save the dataset, semantic arrays and stimulus lookup"

dataset_path = output_path / "teaching_recall.h5"
save_dict_to_hdf5(data, str(dataset_path))
np.save(output_path / "item_features.npy", np.asarray(features))
np.save(output_path / "semantic_similarities.npy", np.asarray(similarities))
np.save(output_path / "semantic_distances.npy", np.asarray(distances))
(output_path / "stimulus_ids.json").write_text(json.dumps(item_ids, indent=2) + "\n")
reloaded = load_data(str(dataset_path))
arrays_preserved = all(np.array_equal(data[name], reloaded[name]) for name in data)
assert arrays_preserved
print("All dataset arrays preserved:", arrays_preserved)
print("Saved files:", sorted(path.name for path in output_path.iterdir()))


All dataset arrays preserved: True
Saved files: ['item_features.npy', 'semantic_distances.npy', 'semantic_similarities.npy', 'stimulus_ids.json', 'teaching_recall.h5']


Supply `teaching_recall.h5` as an analysis input and, for semantic analyses, `semantic_distances.npy` as `distance_matrix_path`. This example's condition field is `condition`, so use queries such as `data['condition'] == 1` rather than the workshop's `list_type`. Keep `stimulus_ids.json` with the semantic arrays when adapting the procedure to another experiment.

[Complete research example: preparing the war-photo-recall data](../downloads/examples/war-photo-data-preparation.ipynb) shows the original workbook reconciliation, scoring exclusions, item mapping and saved provenance. It requires private lab inputs and is provided for inspection; the tutorial above is self-contained. The real example's final feature table contains **emotion ratings**, not semantic associations.

Continue with [From data to model comparison](template-tour.ipynb).